In [26]:
import pandas as pd
from data_analysis_utils import Univariate, Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)



In [36]:
data = pd.read_csv('kidney_disease.csv')
print(data.sample(10))





      id   age    bp     sg   al   su       rbc        pc         pcc          ba    bgr     bu   sc    sod   pot  hemo  pcv    wc   rc  htn   dm cad appet   pe  ane classification
259  259  35.0  80.0  1.020  0.0  0.0    normal    normal  notpresent  notpresent  104.0   31.0  1.2  135.0   5.0  16.1   45  4300  5.2   no   no  no  good   no   no         notckd
338  338  62.0  80.0  1.020  0.0  0.0    normal    normal  notpresent  notpresent  132.0   34.0  0.8  147.0   3.5  17.8   44  4700  4.5   no   no  no  good   no   no         notckd
244  244  64.0  90.0  1.015  3.0  2.0       NaN  abnormal     present  notpresent  463.0   64.0  2.8  135.0   4.1  12.2   40  9800  4.6  yes  yes  no  good   no  yes            ckd
399  399  58.0  80.0  1.025  0.0  0.0    normal    normal  notpresent  notpresent  131.0   18.0  1.1  141.0   3.5  15.8   53  6800  6.1   no   no  no  good   no   no         notckd
337  337  44.0  70.0  1.025  0.0  0.0    normal    normal  notpresent  notpresent   92.0   40.0

In [37]:
print(data.shape)

data.drop(columns = 'id', axis = 1, inplace = True)

quan, qual = Univariate.quanQual(data)

print(quan, qual)


(400, 26)
['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo'] ['rbc', 'pc', 'pcc', 'ba', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [38]:
print(data.isnull().mean()*100)


age                2.25
bp                 3.00
sg                11.75
al                11.50
su                12.25
rbc               38.00
pc                16.25
pcc                1.00
ba                 1.00
bgr               11.00
bu                 4.75
sc                 4.25
sod               21.75
pot               22.00
hemo              13.00
pcv               17.50
wc                26.25
rc                32.50
htn                0.50
dm                 0.50
cad                0.50
appet              0.25
pe                 0.25
ane                0.25
classification     0.00
dtype: float64


1. Drops rows if the missing percentage in a column is tiny (<= 5%).

2. If missing percentage is less (<= to 25%)
    
    Numeric columns:

    1. Uses Mean if distribution is roughly normal.
    2. Uses Median if skewed.

    Categorical columns:

    3. Uses Mode.


In [30]:
low_missing_cols = [col for col in data.columns if 0 < data[col].isna().mean()*100 <= 5]
if low_missing_cols:
    data = data.dropna(subset=low_missing_cols)

# SimpleImputer for >5% & ≤25% missing
mid_missing_num = [col for col in quan if 5 < data[col].isna().mean()*100 <= 25]
mid_missing_cat = [col for col in qual if 5 < data[col].isna().mean()*100 <= 25]

if mid_missing_num:
    mean_cols = [col for col in mid_missing_num if abs(data[col].skew()) < 1]
    median_cols = list(set(mid_missing_num) - set(mean_cols))

    if mean_cols: # mean for normal data
        data.loc[:, mean_cols] = SimpleImputer(strategy='mean').fit_transform(data[mean_cols])
    if median_cols: # median for skewed data
        data.loc[:, median_cols] = SimpleImputer(strategy='median').fit_transform(data[median_cols])

if mid_missing_cat: # mode for categorical data
    data.loc[:, mid_missing_cat] = SimpleImputer(strategy='most_frequent').fit_transform(data[mid_missing_cat])


print(data.shape)

print((data.isnull().mean()*100).round())


(355, 25)
age                0.0
bp                 0.0
sg                 0.0
al                 0.0
su                 0.0
rbc               38.0
pc                 0.0
pcc                0.0
ba                 0.0
bgr                0.0
bu                 0.0
sc                 0.0
sod                0.0
pot                0.0
hemo               0.0
pcv                0.0
wc                25.0
rc                30.0
htn                0.0
dm                 0.0
cad                0.0
appet              0.0
pe                 0.0
ane                0.0
classification     0.0
dtype: float64


3. Model-based (KNN) for remaining missing values

In [32]:
high_missing_cols = [col for col in data.columns if data[col].isna().mean()*100 > 25]

for col in high_missing_cols:
    known = data[data[col].notna()] # data without null rows
    unknown = data[data[col].isna()] # data with null rows
    if unknown.empty:
        continue

    X_known = pd.get_dummies(known.drop(columns=[col]), dummy_na=True)
    X_unknown = pd.get_dummies(unknown.drop(columns=[col]), dummy_na=True)
    X_unknown = X_unknown.reindex(columns=X_known.columns, fill_value=0) # align columns

    y_known = known[col]
    
    if col in quan:
        model = DecisionTreeRegressor(random_state=10)
    else:
        model = DecisionTreeClassifier(random_state=10)

    model.fit(X_known, y_known)
    predicted_value = model.predict(X_unknown)
    
    data.loc[data[col].isna(), col] = predicted_value


In [39]:
data = Preprocessing.simple_missing(data, quan, qual)


In [40]:
data = Preprocessing.model_missing(data, quan)
print(data.shape)

print(data.isnull().mean()*100)


(355, 25)
age               0.0
bp                0.0
sg                0.0
al                0.0
su                0.0
rbc               0.0
pc                0.0
pcc               0.0
ba                0.0
bgr               0.0
bu                0.0
sc                0.0
sod               0.0
pot               0.0
hemo              0.0
pcv               0.0
wc                0.0
rc                0.0
htn               0.0
dm                0.0
cad               0.0
appet             0.0
pe                0.0
ane               0.0
classification    0.0
dtype: float64


In [41]:
data.sample(10)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
353,39.0,60.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,86.0,37.0,0.6,150.0,5.0,13.60000,51,5800,4.5,no,no,no,good,no,no,notckd
69,26.0,70.0,1.015,0.0,4.0,abnormal,normal,notpresent,notpresent,250.0,20.0,1.1,138.0,4.4,15.60000,52,6900,6.0,no,yes,no,good,no,no,ckd
351,29.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,83.0,49.0,0.9,139.0,3.3,17.50000,40,9900,4.7,no,no,no,good,no,no,notckd
183,30.0,70.0,1.015,0.0,0.0,abnormal,normal,notpresent,notpresent,101.0,106.0,6.5,135.0,4.3,12.52524,41,6300,8.0,no,no,no,poor,no,no,ckd
137,45.0,60.0,1.010,2.0,0.0,normal,abnormal,present,notpresent,268.0,86.0,4.0,134.0,5.1,10.00000,29,9200,3.5,yes,yes,no,good,no,no,ckd
233,51.0,100.0,1.015,2.0,0.0,normal,normal,notpresent,present,93.0,20.0,1.6,146.0,4.5,12.52524,41,9800,4.3,no,no,no,poor,no,no,ckd
235,45.0,70.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,113.0,93.0,2.3,138.0,4.4,7.90000,26,5700,4.5,no,no,yes,good,no,yes,ckd
170,66.0,70.0,1.015,2.0,5.0,abnormal,normal,notpresent,notpresent,447.0,41.0,1.7,131.0,3.9,12.50000,33,9600,4.4,yes,yes,no,good,no,no,ckd
149,65.0,70.0,1.020,1.0,0.0,abnormal,abnormal,notpresent,notpresent,139.0,29.0,1.0,138.0,4.4,10.50000,32,2200,3.8,yes,no,no,good,yes,no,ckd
360,35.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,105.0,39.0,0.5,135.0,3.9,14.70000,43,5800,6.2,no,no,no,good,no,no,notckd


In [43]:
data.to_csv("Pre_kidney_disease.csv", index=False)
